# 📊 Neural Network Parameter Analysis Project

## Project Overview
This notebook performs a comprehensive analysis of how different neural network parameters affect model performance. We systematically investigate:

- **Optimizers**: Adam, SGD, AdamW
- **Network Depth**: Shallow (1 layer), Medium (2 layers), Deep (3 layers)
- **Batch Normalization**: With/without BN layers
- **Dropout**: Different dropout rates (0.0, 0.3, 0.5)
- **Hyperparameters**: Learning rate, batch size, weight decay

## Dataset
We use **Fashion-MNIST** - a dataset of 70,000 grayscale images (28x28 pixels) of clothing items across 10 classes. This dataset provides a good balance between complexity and training speed.

## Methodology
All experiments use:
- Fixed random seed (42) for reproducibility
- 10 training epochs (sufficient for convergence analysis)
- Train/validation split (90/10)
- Test set for final evaluation

---

Cell 1: Imports 
# Import all necessary libraries for analysis and visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Set visualization style for professional plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ All libraries imported successfully!")
print(f"📊 Seaborn version: {sns.__version__}")
print(f"📊 Matplotlib version: {plt.matplotlib.__version__}")


## 1. Data Loading and Initial Exploration
First, we load the experimental results from the CSV file generated by `experiments.py`. This contains all 34 experiment configurations and their performance metrics.

Cell 2: Load Data

In [ ]:

# Create results directory if it doesn't exist
os.makedirs('results', exist_ok=True)

# Load results from CSV
try:
    df = pd.read_csv('results/experiment_results.csv')
    print(f"✅ Data loaded successfully!")
    print(f"📊 Total experiments: {len(df)}")
    print(f"📊 Total parameters: {len(df.columns)}")
except FileNotFoundError:
    print("❌ Error: 'experiment_results.csv' not found!")
    print("Please run 'experiments.py' first to generate results.")
    print("Command: python experiments.py")

print("\n📋 Sample of loaded data:")
df.head()

## 2. Data Overview
Let's examine the structure and statistics of our experimental data.

Cell 3: Data Overview

In [ ]:
 # Basic statistics
print("📊 Statistical Summary:")
display(df.describe())

print("\n📊 Missing Values:")
display(df.isnull().sum())

print("\n📊 Data Types:")
display(df.dtypes)

print("\n📊 Unique values per categorical column:")
categorical_cols = ['category', 'optimizer', 'layers', 'use_batchnorm']
for col in categorical_cols:
    if col in df.columns:
        print(f"  {col}: {df[col].nunique()} unique values")
        print(f"    Values: {df[col].unique()}")

## 3. Distribution of Experiment Categories
Understanding how our experiments are distributed across different parameter groups.

 Cell 4: Experiment Distribution

In [ ]:
plt.figure(figsize=(12, 5))

# Count of experiments per category
plt.subplot(1, 2, 1)
category_counts = df['category'].value_counts()
sns.barplot(x=category_counts.index, y=category_counts.values, palette='viridis')
plt.title('Number of Experiments per Category', fontsize=14, fontweight='bold')
plt.xlabel('Parameter Category')
plt.ylabel('Number of Experiments')
for i, v in enumerate(category_counts.values):
    plt.text(i, v + 0.1, str(v), ha='center', va='bottom')

# Accuracy distribution
plt.subplot(1, 2, 2)
sns.histplot(data=df, x='test_acc', bins=15, kde=True, color='skyblue')
plt.title('Distribution of Test Accuracies', fontsize=14, fontweight='bold')
plt.xlabel('Test Accuracy (%)')
plt.ylabel('Frequency')
plt.axvline(df['test_acc'].mean(), color='red', linestyle='--', label=f'Mean: {df["test_acc"].mean():.2f}%')
plt.axvline(df['test_acc'].median(), color='green', linestyle='--', label=f'Median: {df["test_acc"].median():.2f}%')
plt.legend()

plt.tight_layout()
plt.savefig('results/experiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()  

## 4. Optimizer Analysis
**Goal:** Compare the performance of different optimization algorithms on networks of varying depth.

**Hypotheses:**
- Adam and AdamW should converge faster and achieve better accuracy than SGD
- The advantage of adaptive optimizers should be more pronounced in deeper networks
- SGD with momentum might still achieve competitive accuracy with proper tuning

Cell 5: Optimizer Analysis

In [ ]:
 
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy by Optimizer
ax1 = axes[0, 0]
sns.barplot(data=df[df['category']=='optimizer'], x='optimizer', y='test_acc', 
            hue='layers', palette='Set2', ax=ax1)
ax1.set_title('Accuracy by Optimizer Type and Network Depth', fontsize=14, fontweight='bold')
ax1.set_xlabel('Optimizer')
ax1.set_ylabel('Test Accuracy (%)')
ax1.legend(title='Network Depth', loc='lower right')
ax1.grid(True, alpha=0.3)

# Plot 2: Training Time by Optimizer
ax2 = axes[0, 1]
sns.barplot(data=df[df['category']=='optimizer'], x='optimizer', y='train_time', 
            hue='layers', palette='Set2', ax=ax2)
ax2.set_title('Training Time by Optimizer Type', fontsize=14, fontweight='bold')
ax2.set_xlabel('Optimizer')
ax2.set_ylabel('Time (seconds)')
ax2.legend(title='Network Depth', loc='upper left')
ax2.grid(True, alpha=0.3)

# Plot 3: Optimizer Performance vs Depth
ax3 = axes[1, 0]
optimizer_depth = df[df['category']=='optimizer'].pivot_table(
    index='layers', columns='optimizer', values='test_acc', aggfunc='mean'
)
optimizer_depth.plot(kind='bar', ax=ax3, colormap='Set2')
ax3.set_title('Accuracy vs Network Depth per Optimizer', fontsize=14, fontweight='bold')
ax3.set_xlabel('Network Depth')
ax3.set_ylabel('Mean Test Accuracy (%)')
ax3.legend(title='Optimizer')
ax3.grid(True, alpha=0.3)

# Plot 4: Performance ratio (compared to Adam)
ax4 = axes[1, 1]
baseline = df[(df['category']=='optimizer') & (df['optimizer']=='adam')].groupby('layers')['test_acc'].mean()
optimizer_perf = df[df['category']=='optimizer'].groupby(['optimizer', 'layers'])['test_acc'].mean().unstack()
for opt in optimizer_perf.index:
    if opt != 'adam':
        ratio = optimizer_perf.loc[opt] / baseline * 100
        ratio.plot(kind='bar', ax=ax4, label=f'{opt} / Adam (%)', alpha=0.7)
ax4.set_title('Relative Performance vs Adam (baseline = 100%)', fontsize=14, fontweight='bold')
ax4.set_xlabel('Network Depth')
ax4.set_ylabel('Performance Ratio (%)')
ax4.axhline(100, color='red', linestyle='--', alpha=0.5)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/optimizer_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical Summary
print("📊 Optimizer Performance Summary:")
print(df[df['category']=='optimizer'].groupby('optimizer')['test_acc'].agg(['mean', 'std', 'min', 'max'])) 

## 5. Batch Normalization Analysis
**Goal:** Evaluate the impact of Batch Normalization on training and performance.

**Hypotheses:**
- BN should improve convergence speed and accuracy
- BN should be more beneficial for deeper networks
- BN may reduce sensitivity to initialization and learning rate

Cell 6: BatchNorm Analysis

In [ ]:
 
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy by BatchNorm
ax1 = axes[0, 0]
sns.barplot(data=df[df['category']=='batchnorm'], x='use_batchnorm', y='test_acc', 
            hue='layers', palette='Set3', ax=ax1)
ax1.set_title('BatchNorm Impact on Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Batch Normalization')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_xticklabels(['Without BN', 'With BN'])
ax1.legend(title='Network Depth')
ax1.grid(True, alpha=0.3)

# Plot 2: Training Time with BatchNorm
ax2 = axes[0, 1]
sns.barplot(data=df[df['category']=='batchnorm'], x='use_batchnorm', y='train_time', 
            hue='layers', palette='Set3', ax=ax2)
ax2.set_title('Training Time with BatchNorm', fontsize=14, fontweight='bold')
ax2.set_xlabel('Batch Normalization')
ax2.set_ylabel('Time (seconds)')
ax2.set_xticklabels(['Without BN', 'With BN'])
ax2.legend(title='Network Depth')
ax2.grid(True, alpha=0.3)

# Plot 3: Accuracy comparison by depth
ax3 = axes[1, 0]
bn_comparison = df[df['category']=='batchnorm'].pivot_table(
    index='layers', columns='use_batchnorm', values='test_acc', aggfunc='mean'
)
bn_comparison.plot(kind='bar', ax=ax3, colormap='coolwarm')
ax3.set_title('Accuracy Comparison: With vs Without BatchNorm', fontsize=14, fontweight='bold')
ax3.set_xlabel('Network Depth')
ax3.set_ylabel('Mean Test Accuracy (%)')
ax3.set_xticklabels(['Without BN', 'With BN'])
ax3.legend(title='BatchNorm')
ax3.grid(True, alpha=0.3)

# Plot 4: Accuracy improvement
ax4 = axes[1, 1]
improvement = bn_comparison[True] - bn_comparison[False]
improvement.plot(kind='bar', ax=ax4, color='green')
ax4.set_title('Accuracy Improvement with BatchNorm', fontsize=14, fontweight='bold')
ax4.set_xlabel('Network Depth')
ax4.set_ylabel('Accuracy Improvement (%)')
ax4.axhline(0, color='red', linestyle='--', alpha=0.5)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/batchnorm_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical Summary
print("📊 BatchNorm Performance Summary:")
print(df[df['category']=='batchnorm'].groupby('use_batchnorm')['test_acc'].agg(['mean', 'std']))
print("\n📊 Improvement by depth:")
print(improvement)

## 6. Dropout Analysis
**Goal:** Determine the optimal dropout rate for different network depths.

**Hypotheses:**
- Dropout should improve generalization (especially in deeper networks)
- Optimal dropout rate increases with network depth
- Too much dropout (>0.5) may hurt performance

Cell 7: Dropout Analysis

In [ ]:


fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Accuracy by Dropout Rate
ax1 = axes[0, 0]
sns.lineplot(data=df[df['category']=='dropout'], x='dropout_rate', y='test_acc', 
             hue='layers', marker='o', linewidth=2.5, ax=ax1)
ax1.set_title('Accuracy vs Dropout Rate by Network Depth', fontsize=14, fontweight='bold')
ax1.set_xlabel('Dropout Rate')
ax1.set_ylabel('Test Accuracy (%)')
ax1.legend(title='Network Depth')
ax1.grid(True, alpha=0.3)

# Plot 2: Optimal Dropout Rate
ax2 = axes[0, 1]
best_dropout = df[df['category']=='dropout'].groupby(['layers', 'dropout_rate'])['test_acc'].mean().reset_index()
best_dropout = best_dropout.loc[best_dropout.groupby('layers')['test_acc'].idxmax()]
sns.barplot(data=best_dropout, x='layers', y='dropout_rate', palette='coolwarm', ax=ax2)
ax2.set_title('Optimal Dropout Rate per Network Depth', fontsize=14, fontweight='bold')
ax2.set_xlabel('Network Depth')
ax2.set_ylabel('Optimal Dropout Rate')
ax2.axhline(0.3, color='red', linestyle='--', alpha=0.5, label='Common choice: 0.3')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Dropout vs No Dropout
ax3 = axes[1, 0]
dropout_effect = df[df['category']=='dropout'].pivot_table(
    index='layers', columns='dropout_rate', values='test_acc', aggfunc='mean'
)
dropout_effect[[0.0, 0.3, 0.5]].plot(kind='bar', ax=ax3, colormap='viridis')
ax3.set_title('Accuracy with Different Dropout Rates', fontsize=14, fontweight='bold')
ax3.set_xlabel('Network Depth')
ax3.set_ylabel('Mean Test Accuracy (%)')
ax3.legend(title='Dropout Rate')
ax3.grid(True, alpha=0.3)

# Plot 4: Overfitting indicator (gap between train and validation)
ax4 = axes[1, 1]
# Note: Since we don't have train acc directly, we use validation accuracy
dropout_best = df[(df['category']=='dropout') & (df['dropout_rate'].isin([0.0, 0.3, 0.5]))]
sns.lineplot(data=dropout_best, x='dropout_rate', y='best_val_acc', 
             hue='layers', marker='s', linewidth=2, ax=ax4)
ax4.set_title('Validation Accuracy by Dropout Rate', fontsize=14, fontweight='bold')
ax4.set_xlabel('Dropout Rate')
ax4.set_ylabel('Best Validation Accuracy (%)')
ax4.legend(title='Network Depth')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/dropout_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical Summary
print("📊 Dropout Performance Summary:")
print(df[df['category']=='dropout'].groupby('dropout_rate')['test_acc'].agg(['mean', 'std']))

## 7. Hyperparameter Tuning Analysis
**Goal:** Find the optimal combination of hyperparameters (learning rate, batch size, weight decay).

**Hypotheses:**
- Learning rate is the most critical hyperparameter
- Smaller batch sizes may lead to better generalization
- Weight decay helps prevent overfitting

Cell 8: Hyperparameter Tuning Analysis

In [ ]:
 

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

hp_data = df[df['category']=='hp_tuning']

# Plot 1: Learning Rate Impact
ax1 = axes[0, 0]
scatter = ax1.scatter(hp_data['learning_rate'], hp_data['test_acc'], 
                     s=hp_data['batch_size']/2, 
                     c=hp_data['weight_decay'], 
                     cmap='viridis', alpha=0.7)
ax1.set_xscale('log')
ax1.set_title('Learning Rate and Batch Size Impact on Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Learning Rate (log scale)')
ax1.set_ylabel('Test Accuracy (%)')
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Weight Decay')
ax1.grid(True, alpha=0.3)

# Plot 2: Batch Size Impact
ax2 = axes[0, 1]
sns.boxplot(data=hp_data, x='batch_size', y='test_acc', ax=ax2)
ax2.set_title('Batch Size Impact on Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Test Accuracy (%)')
ax2.grid(True, alpha=0.3)

# Plot 3: Best vs Worst Configurations
ax3 = axes[1, 0]
best_idx = hp_data['test_acc'].idxmax()
worst_idx = hp_data['test_acc'].idxmin()
comparison = pd.DataFrame({
    'Best': hp_data.loc[best_idx, ['learning_rate', 'batch_size', 'weight_decay']],
    'Worst': hp_data.loc[worst_idx, ['learning_rate', 'batch_size', 'weight_decay']]
})
comparison.plot(kind='bar', ax=ax3, color=['green', 'red'], alpha=0.8)
ax3.set_title('Best vs Worst Hyperparameter Configuration', fontsize=14, fontweight='bold')
ax3.set_xlabel('Parameters')
ax3.set_ylabel('Value')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
ax3.legend(title='Configuration')
ax3.grid(True, alpha=0.3)

# Plot 4: Performance Ranking
ax4 = axes[1, 1]
hp_sorted = hp_data.sort_values('test_acc', ascending=False).reset_index(drop=True)
sns.barplot(data=hp_sorted, x='learning_rate', y='test_acc', 
            hue='batch_size', dodge=False, ax=ax4)
ax4.set_title('Performance Ranking of Hyperparameter Configurations', fontsize=14, fontweight='bold')
ax4.set_xlabel('Configuration (by Learning Rate)')
ax4.set_ylabel('Test Accuracy (%)')
ax4.legend(title='Batch Size')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/hp_tuning_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Best Configuration:")
best_config = hp_data.loc[hp_data['test_acc'].idxmax()]
print(best_config[['learning_rate', 'batch_size', 'weight_decay', 'test_acc']])

print("\n📊 Worst Configuration:")
worst_config = hp_data.loc[hp_data['test_acc'].idxmin()]
print(worst_config[['learning_rate', 'batch_size', 'weight_decay', 'test_acc']])

print("\n📊 Configuration Ranking:")
print(hp_data[['experiment_id', 'learning_rate', 'batch_size', 'weight_decay', 'test_acc']].sort_values('test_acc', ascending=False).head(5))

## 8. Comprehensive Correlation Analysis
**Goal:** Understand the relationships between different parameters and model performance.

Cell 9: Correlation Analysis

In [ ]:
 

plt.figure(figsize=(14, 10))

# Select numerical features for correlation
features = ['test_acc', 'train_time', 'dropout_rate', 'use_batchnorm']
if 'learning_rate' in df.columns:
    features.extend(['learning_rate', 'batch_size', 'weight_decay'])

# Create correlation matrix
correlation_matrix = df[features].corr()

# Plot heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', linewidths=2, linecolor='white',
            cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Parameters and Performance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Print strongest correlations with test accuracy
print("📊 Strongest Correlations with Test Accuracy:")
correlations = correlation_matrix['test_acc'].sort_values(ascending=False)
for feature, corr in correlations.items():
    if feature != 'test_acc':
        print(f"  {feature}: {corr:.3f}")

## 9. Overall Performance Summary
**Goal:** Identify the best performing configuration and key insights.

Cell 10: Final Summary

In [ ]:
 

print("="*80)
print("📊 PROJECT FINAL SUMMARY")
print("="*80)

print(f"\n✅ Best Overall Accuracy: {df['test_acc'].max():.2f}%")
print(f"   Experiment #{df[df['test_acc']==df['test_acc'].max()]['experiment_id'].values[0]}")
print(f"   Configuration:")
best_exp = df.loc[df['test_acc'].idxmax()]
for col in best_exp.index:
    if col not in ['experiment_id', 'category', 'test_acc', 'test_loss', 'train_time', 'best_val_acc']:
        print(f"     - {col}: {best_exp[col]}")

print(f"\n⏱️  Fastest Training Time: {df['train_time'].min():.2f} seconds")
print(f"   Experiment #{df[df['train_time']==df['train_time'].min()]['experiment_id'].values[0]}")

print(f"\n⚡ Best Accuracy / Time Tradeoff:")
best_tradeoff = df.loc[(df['test_acc'] / df['train_time']).idxmax()]
print(f"   Experiment #{best_tradeoff['experiment_id']}")
print(f"   Accuracy: {best_tradeoff['test_acc']:.2f}%")
print(f"   Time: {best_tradeoff['train_time']:.2f}s")
print(f"   Efficiency: {best_tradeoff['test_acc']/best_tradeoff['train_time']:.4f} %/s")

print("\n" + "="*80)
print("📈 KEY INSIGHTS")
print("="*80)

# Generate insights
print("\n1. OPTIMIZER INSIGHTS:")
opt_summary = df[df['category']=='optimizer'].groupby('optimizer')['test_acc'].mean()
best_opt = opt_summary.idxmax()
print(f"   - Best optimizer: {best_opt} ({opt_summary.max():.2f}%)")
print(f"   - Worst optimizer: {opt_summary.idxmin()} ({opt_summary.min():.2f}%)")

print("\n2. DEPTH INSIGHTS:")
depth_summary = df.groupby('layers')['test_acc'].mean()
print(f"   - Best depth: {depth_summary.idxmax()} ({depth_summary.max():.2f}%)")
print(f"   - Worst depth: {depth_summary.idxmin()} ({depth_summary.min():.2f}%)")

print("\n3. BATCH NORM INSIGHTS:")
bn_summary = df[df['category']=='batchnorm'].groupby('use_batchnorm')['test_acc'].mean()
if bn_summary[True] > bn_summary[False]:
    print(f"   - BatchNorm improves accuracy by {bn_summary[True] - bn_summary[False]:.2f}%")
else:
    print(f"   - BatchNorm decreases accuracy by {bn_summary[False] - bn_summary[True]:.2f}%")

print("\n4. DROPOUT INSIGHTS:")
dropout_summary = df[df['category']=='dropout'].groupby('dropout_rate')['test_acc'].mean()
best_drop = dropout_summary.idxmax()
print(f"   - Best dropout rate: {best_drop} ({dropout_summary.max():.2f}%)")
print(f"   - No dropout gives: {dropout_summary[0.0]:.2f}%")

print("\n5. HYPERPARAMETER INSIGHTS:")
if 'learning_rate' in df.columns:
    hp_best = df[df['category']=='hp_tuning'].loc[df['test_acc'].idxmax()]
    print(f"   - Best learning rate: {hp_best['learning_rate']}")
    print(f"   - Best batch size: {hp_best['batch_size']}")
    print(f"   - Best weight decay: {hp_best['weight_decay']}")

print("\n" + "="*80)
print("🎯 RECOMMENDED CONFIGURATION")
print("="*80)
print("Based on the analysis, the recommended configuration is:")
print(f"  • Optimizer: {best_exp['optimizer']}")
print(f"  • Network Depth: {best_exp['layers']}")
print(f"  • Batch Normalization: {best_exp['use_batchnorm']}")
print(f"  • Dropout Rate: {best_exp['dropout_rate']}")
if 'learning_rate' in best_exp.index:
    print(f"  • Learning Rate: {best_exp['learning_rate']}")
    print(f"  • Batch Size: {best_exp['batch_size']}")
    print(f"  • Weight Decay: {best_exp['weight_decay']}")
print(f"\n✅ Expected Accuracy: {best_exp['test_acc']:.2f}%")
print(f"⏱️  Training Time: {best_exp['train_time']:.2f} seconds")

print("\n" + "="*80)
print("📊 PROJECT COMPLETED SUCCESSFULLY! 🎉")
print("="*80)

## 10. References and Future Work

### References
1. Kingma, D. P., & Ba, J. (2014). Adam: A Method for Stochastic Optimization. arXiv:1412.6980.
2. Ioffe, S., & Szegedy, C. (2015). Batch Normalization: Accelerating Deep Network Training. ICML.
3. Srivastava, N., et al. (2014). Dropout: A Simple Way to Prevent Neural Networks from Overfitting. JMLR.
4. Xiao, H., Rasul, K., & Vollgraf, R. (2017). Fashion-MNIST: A Novel Image Dataset for Machine Learning. arXiv:1708.07747.

### Future Improvements
- Experiment with deeper networks (5+ layers)
- Test on more complex datasets (CIFAR-10, ImageNet subset)
- Implement learning rate scheduling
- Use Bayesian optimization for hyperparameter tuning
- Analyze loss landscapes
- Implement regularization techniques (L1/L2, weight decay variants)

---
**Project completed on:** {date}
**Total runtime:** {runtime}
**Author:** Neural Network Research Team